# BO Manual Evaluation Overview (Week 12)

This notebook aggregates manually evaluated weekly points from all function notebooks (`week_12_function_1..8_analysis.ipynb`), computes running best performance per function, and highlights evaluations that achieved a new maximum.

In [4]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Resolve repo root robustly even if notebook is run from subfolders
cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "notebooks").exists() and (p / "initial_data").exists():
        repo_root = p
        break

if repo_root is None:
    raise RuntimeError("Could not locate repository root (expected folders: notebooks/ and initial_data/).")

week12_dir = repo_root / "notebooks" / "week_12"
files = sorted(week12_dir.glob("week_12_function_*_analysis.ipynb"))
if not files:
    raise RuntimeError(f"No week_12 function notebooks found in: {week12_dir}")

pattern_x = re.compile(r"X_new_point_week_(\d+)\s*=\s*np\.array\(\s*\[\s*\[(.*?)\]\s*\]\s*\)", re.S)
pattern_y = re.compile(r"y_new_point_week_(\d+)\s*=\s*np\.array\(\s*\[(.*?)\]\s*\)", re.S)


def parse_float_list(text):
    parts = [p.strip() for p in text.replace("\n", " ").split(",") if p.strip()]
    return [float(p) for p in parts]


rows = []
for fp in files:
    mfunc = re.search(r"function_(\d+)_analysis", fp.name)
    if not mfunc:
        continue
    fn = int(mfunc.group(1))

    nb = json.loads(fp.read_text(encoding="utf-8"))
    code_text = "\n".join(
        "".join(c.get("source", []))
        for c in nb.get("cells", [])
        if c.get("cell_type") == "code"
    )

    x_map, y_map = {}, {}
    for wk, payload in pattern_x.findall(code_text):
        x_map[int(wk)] = parse_float_list(payload)
    for wk, payload in pattern_y.findall(code_text):
        vals = parse_float_list(payload)
        if vals:
            y_map[int(wk)] = vals[0]

    common_weeks = sorted(set(x_map).intersection(y_map))
    running_max = None
    for wk in common_weeks:
        yv = float(y_map[wk])
        running_max = yv if running_max is None else max(running_max, yv)
        is_new_max = abs(yv - running_max) < 1e-12
        rows.append(
            {
                "Function": f"Function {fn}",
                "FunctionID": fn,
                "Week": wk,
                "Point": ", ".join(f"{v:.6f}" for v in x_map[wk]),
                "y": yv,
                "New_Max": is_new_max,
            }
        )

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError(
        "No manual weekly evaluation points were parsed. "
        "Check that week_12 notebooks contain X_new_point_week_* and y_new_point_week_* variables."
    )

df = df.sort_values(["FunctionID", "Week"]).reset_index(drop=True)

# Load baseline (initial_data) outputs so ranks are relative to full history, not only weekly manual points
baseline_y_by_fn = {}
for fn in sorted(df["FunctionID"].unique()):
    yp = repo_root / "initial_data" / f"function_{fn}" / "initial_outputs.npy"
    if yp.exists():
        baseline_y_by_fn[fn] = np.load(yp).astype(float).tolist()
    else:
        baseline_y_by_fn[fn] = []


def rolling_rank_with_baseline(values, baseline_values):
    # Rank each weekly point against (initial data + weekly points up to that week)
    seen = list(baseline_values)
    ranks = []
    for v in values:
        seen.append(v)
        ranks.append(1 + sum(prev > v for prev in seen))  # maximize y
    return ranks


def assign_ranks(group):
    fn = int(group["FunctionID"].iloc[0])
    ranks = rolling_rank_with_baseline(group["y"].tolist(), baseline_y_by_fn.get(fn, []))
    return pd.Series(ranks, index=group.index)


df["Rank_At_Week"] = df.groupby("FunctionID", group_keys=False).apply(assign_ranks).astype(int)

y_matrix = df.pivot(index="Function", columns="Week", values="y").round(4)
mask_new = (
    df.pivot(index="Function", columns="Week", values="New_Max")
    .reindex_like(y_matrix)
    .fillna(False)
)

rank_matrix = df.pivot(index="Function", columns="Week", values="Rank_At_Week")


def highlight_new_max(_):
    styles = pd.DataFrame("", index=y_matrix.index, columns=y_matrix.columns)
    styles[mask_new] = "background-color: #c6efce; font-weight: 600;"
    return styles


print("Manual evaluation y-matrix (green = new max at that week)")
try:
    display(y_matrix.style.apply(highlight_new_max, axis=None))
except AttributeError:
    # Fallback if jinja2 is unavailable: plain table without color
    display(y_matrix)

print("\nRank matrix of weekly evaluated points (ranked against initial data + weekly history; 1 = best-so-far)")
display(rank_matrix)

summary = (
    df.groupby(["Function", "FunctionID"], as_index=False)
    .agg(
        Evaluations=("Week", "count"),
        New_Max_Count=("New_Max", "sum"),
        Best_y=("y", "max"),
    )
    .sort_values("FunctionID")
)
summary["New_Max_Rate_%"] = (100 * summary["New_Max_Count"] / summary["Evaluations"]).round(1)
summary = summary[["Function", "Evaluations", "New_Max_Count", "New_Max_Rate_%", "Best_y"]]

print("\nBO success overview by function (how often a weekly point set a new max):")
display(summary)

overall_rate = 100.0 * df["New_Max"].sum() / len(df)
print(f"Overall: {df['New_Max'].sum()}/{len(df)} weekly evaluations were new maxima ({overall_rate:.1f}%).")

Manual evaluation y-matrix (green = new max at that week)


C:\Users\gophi\AppData\Local\Temp\ipykernel_51956\3262174900.py:108: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["Rank_At_Week"] = df.groupby("FunctionID", group_keys=False).apply(assign_ranks).astype(int)


Week,1,2,3,4,5,6,7,8,9,10,11
Function,,,,,,,,,,,
Function 1,0.0256,-0.0082,0.0000,0.0848,0.0000,0.3589,0.5158,1.7583,0.8362,0.3690,0.9569
Function 2,0.4588,0.4688,0.5521,-0.0485,0.4712,0.3100,0.6703,0.6255,0.7457,0.6416,0.6362
Function 3,-0.0118,-0.0961,-0.0617,-0.0461,-0.0053,-0.0064,-0.0152,-0.0073,-0.0079,-0.0048,-0.0150
Function 4,-11.5514,-0.0580,-0.0140,-0.1000,0.5518,0.3711,-0.0502,0.2620,-1.3668,0.3216,0.1888
Function 5,1086.3645,1935.0093,2066.6745,2323.4366,2748.8300,2981.5518,3136.4200,3327.4292,4228.1174,4460.2266,4462.8873
Function 6,-0.6776,-0.6699,-0.6254,-0.6177,-0.4432,-1.3625,-0.6509,-0.6441,-0.4351,-0.3265,-0.1801
Function 7,0.0345,1.3138,1.6455,1.0249,0.7102,1.7805,1.7220,1.7852,1.8559,2.1028,1.9004
Function 8,9.7436,9.7300,9.6549,9.7246,9.7497,9.7271,9.7489,9.7505,9.7395,9.7117,9.7117



Rank matrix of weekly evaluated points (ranked against initial data + weekly history; 1 = best-so-far)


Week,1,2,3,4,5,6,7,8,9,10,11
Function,,,,,,,,,,,
Function 1,1,12,2,1,3,1,1,1,2,4,2
Function 2,3,3,2,13,4,8,1,2,1,3,4
Function 3,1,9,7,4,1,2,4,3,4,1,7
Function 4,5,1,1,3,1,2,4,3,8,3,5
Function 5,2,1,1,1,1,1,1,1,1,1,1
Function 6,1,1,1,1,1,16,4,4,1,1,1
Function 7,20,2,1,4,5,1,2,1,1,1,2
Function 8,1,2,3,3,1,4,2,1,5,9,9



BO success overview by function (how often a weekly point set a new max):


,Function,Evaluations,New_Max_Count,New_Max_Rate_%,Best_y
0,Function 1,11,5,45.5,1.758258
1,Function 2,11,5,45.5,0.745718
2,Function 3,11,3,27.3,-0.004817
3,Function 4,11,4,36.4,0.551843
4,Function 5,11,11,100.0,4462.887304
5,Function 6,11,8,72.7,-0.180058
6,Function 7,11,7,63.6,2.102801
7,Function 8,11,3,27.3,9.750453


Overall: 46/88 weekly evaluations were new maxima (52.3%).
